## Computer Vision Using Transfer Learning

This notebook allows us to process images of areas hit by disasters in order to see where was most heavily impacted. The model we're pulling in is called YOLO (You Only Look Once). There's many versions of this model, and many hyperparameters we can tinker with in order to adjust how it's performing. As you'll learn throughout the Block, we may need to try many things to see what gives us the best result. We can intuit that newer versions of the model may perform better in general, but perhaps something in the architecture of previous iterations would have performed better in our particular use case. 

Here's a link to the website outlining the different models created/potentially available to you: https://docs.ultralytics.com/models/

To start, I've gone sort of middle of the road for you, we're pulling in YOLOv8. Many of the other models available should work with a simple change to the line of code where we actually import the model. Some may take just a bit of tinkering to our data files for compatability.

*note, the documentation for these models varies a bit in its quality. You may need to do some digging in their website to find specific points of interest. YOLOv8 docs: https://docs.ultralytics.com/models/yolov8/




In [2]:
import ultralytics
from ultralytics import YOLO
import os
import torch
import torch.distributed as dist
import ssl
from pathlib import Path
TRAIN = True

ModuleNotFoundError: No module named 'ultralytics'

Checking to see if we have a bit more power available. If you're using our silver HP laptops, you don't.

In [ ]:
device = 'cpu' if not torch.cuda.is_available() else 'cuda'

Looking to see where the current working directory is. This is good information to have for when we start looking for where our model is getting trained and saved.

In [ ]:
cwd = os.getcwd()
print(cwd)

This is where we're actually pulling in the model. We first allow a connection to where the model lives online and then we download it to a folder in our cwd.

In [ ]:
print('test')

In [ ]:
ssl._create_default_https_context = ssl._create_unverified_context # this one'll let us actually pull in the model
model = YOLO("yolov8n.pt")

Creating some relative paths so that the model saves stuff where it's easy to find. Feel free to mess with this, but know that stuff may not go where I can help you find it

In [ ]:
yaml_loc = "X_View/datasets/data/data.yaml"
full_path = os.path.join(cwd, yaml_loc)
folder_loc = "X_View/datasets/data/"
project_path = os.path.join(cwd, folder_loc)

This is where we're doing our training. As long as you haven't flayed the folder we gave you, this should work pretty smoothly. Take note of the arugements in the model.train function call. We need to tell it where the data lives, how much training we want to conduct, the path for the project, and what we want it to call the folder where we put the trained version of the model. These are things you may consider editing. Just be aware of what the changes you make do to the training process. 

*note, the arguements I've placed here for you are not the only ones that you may want to consider

In [ ]:
if TRAIN:
    results = model.train(
            data=full_path,  # path to data.yaml
            epochs=2,
            project=project_path,
            name="./Xview_mods/Trial", # name of the output folder,
            device=device
        )

### Model Inferencing

So you've trained a model. Cool, I guess. Is it any good? Below is some code to help you apply the model to our validation data. Note that we've previously talked about train/validation/test splits. I haven't given you specific test data. If you take problem with that, we can adjust the data to create a new set. For simplicity's sake, I haven't done it ahead of time. 

Really what we're looking to get out of the code below is some amount of information about how the model's performing. Training a model is easy, making it usable is tricky.

The first half of the cell is code to load in the saved version of the model. You may need to change the "model_loc" to direct to the model you actually want to use. Everything else should *just work* (yeah, right). The key piece of this whole thing (assuming model training went okay-ish) is to tinker with the predict function. **YOU** have control over how the model is making and displaying its predictions. Pay attention to the parameters plugged in there, specifically the conf, iou, and save_txt arguements. These allow you to adjust what we count as a "prediction". Feel free to look up what these are doing, or just try a little bit of sensitivity analysis on adjusting the values. 

In [ ]:
# Path to your trained weights
model_loc = "Xview_mods/Trial/weights/best.pt"
model_path = os.path.join(cwd, folder_loc, model_loc)
val_images_dir = os.path.join(cwd, folder_loc, "val/images")

# Where to save the annotated images
output_proj = "runs"   

model_path = Path(model_path)
val_dir = Path(val_images_dir)

if not model_path.is_file():
    raise FileNotFoundError(f"Model weights not found at: {model_path}")

if not val_dir.is_dir():
    raise NotADirectoryError(f"Validation image folder not found at: {val_dir}")

print(f"Using model: {model_path}")
print(f"Running on validation images in: {val_dir}")

# Load model
model = YOLO(str(model_path))

# Run prediction on all images in the validation folder
results = model.predict(
    source=str(val_dir),  # folder of images
    conf=0.25,
    iou=0.9,
    save=True,            # save annotated images
    save_txt=False,       # set True if you also want YOLO txt outputs
    project=output_proj,
    exist_ok=True,        # don't error if folder exists
    verbose=True,
#     agnostic_nms=True, 
#     visualize=True,
)
